### Statistical Analysis

In [14]:
import pandas as pd
import numpy as np
from scipy import stats

combined_df = pd.read_csv("outputs/degradation_results.csv")
print("Data loaded")
print("Total rows:", len(combined_df))
print()

def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    margin = se* stats.t.ppf((1 + confidence)/2, n-1)

    lower = round(mean- margin, 4)
    upper = round(mean+ margin, 4)
    return mean, lower, upper

print("="*65)
print(f"{'CONFIDENCE INTERVALS -MEAN CONFIDENCE (95%)':^65}")
print("="*65)
print(f"{'Model':<8} {'Condition':<18}   {'Mean':<10} {'Lower':<10} {'Upper'}")
print("-" * 65)

models = ["FP32", "FP16", "INT8"]
conditions = ["clean","blur_sigma1","blur_sigma2","blur_sigma3","jpeg_q90","jpeg_q70","jpeg_q50","gaussian_noise"]

for model in models:
    for condition in conditions:
        subset = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == condition)]

        if len(subset) == 0:
            continue
        mean, lower, upper = confidence_interval(subset["confidence"])
        print(f"{model:<8}  {condition:<18} {round(mean,4):<10} {lower:<10} {upper}")
    print()

print("="*65)

Data loaded
Total rows: 69714

           CONFIDENCE INTERVALS -MEAN CONFIDENCE (95%)           
Model    Condition            Mean       Lower      Upper
-----------------------------------------------------------------
FP32      clean              0.3183     0.3095     0.3271
FP32      blur_sigma1        0.5661     0.5597     0.5725
FP32      blur_sigma2        0.5552     0.5486     0.5618
FP32      blur_sigma3        0.5357     0.5284     0.5429
FP32      jpeg_q90           0.5647     0.5584     0.571
FP32      jpeg_q70           0.5636     0.5572     0.5699
FP32      jpeg_q50           0.5581     0.5517     0.5644
FP32      gaussian_noise     0.5137     0.5073     0.52

FP16      clean              0.5712     0.5649     0.5775
FP16      blur_sigma1        0.5691     0.5627     0.5756
FP16      blur_sigma2        0.5573     0.5506     0.5639
FP16      blur_sigma3        0.5365     0.5289     0.544
FP16      jpeg_q90           0.5703     0.564      0.5766
FP16      jpeg_q70          

In [15]:
# CONFIDENCE INTERVAL FINDINGS
#
# All 95% confidence intervals are narrow (width ~0.006-0.009)
# This means results are highly reliable and repeatable.
#
# Clean video: intervals for FP32, FP16, INT8 do not overlap at all
# FP32=[0.3095, 0.3271], FP16=[0.5649, 0.5775], INT8=[0.6949, 0.7048]
# This statistically confirms the three models have genuinely different
# confidence behaviour on clean video.
#
# Degraded conditions: intervals overlap heavily across all three models
# This statistically confirms confidence convergence under degradation.
# The overconfidence gap between models disappears under real world conditions.

In [29]:
from scipy.stats import mannwhitneyu
import pandas as pd

fp32_clean = combined_df[(combined_df["model"] == "FP32") & (combined_df["condition"] == "clean")]["confidence"]
fp16_clean = combined_df[(combined_df["model"] == "FP16") & (combined_df["condition"] == "clean")]["confidence"]
int8_clean = combined_df[(combined_df["model"] == "INT8") & (combined_df["condition"] == "clean")]["confidence"]

def rank_biserial(group1, group2):
    stat, p = mannwhitneyu(group1, group2)
    n1 = len(group1)
    n2 = len(group2)
    r = 1 -(2*stat) / (n1*n2)
    return round(abs(r), 4), round(p,6)

def interpret_effect(r):
    if r>= 0.5:
        return "LARGE"
    elif r>= 0.3:
        return "MEDIUM"
    else:
        return "SMALL"

print("="* 60)
print(f"{'EFFECT SIZES - CLEAN VIDEO CONFIDENCE':^60}")
print("="*60)
print(f"{'Comparison':<25} {'Effect Size':<15} {'Magnitude':<12} {'p-value'}")
print("="*60)

pairs = [
    ("FP32 vs FP16", fp32_clean, fp16_clean),
    ("FP32 vs INT8", fp32_clean, int8_clean),
    ("FP16 vs INT8", fp16_clean, int8_clean),
]

for name, g1, g2, in pairs:
    r, p = rank_biserial(g1, g2)
    magnitude = interpret_effect(r)
    print(f"{name:<25} {r:<15} {magnitude:<12} {p}")

print("="*60)
print()

print("="*60)
print(f"{'EFFECT SIZES - CLEAN vs DEGRADED (per model)':^60}")
print("="*60)
print(f"{'Comparison':<30} {'Effect Size':<15} {'Magnitude'}")
print("-"*60)

for model in["FP32", "FP16", "INT8"]:
    clean = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "clean")]["confidence"]
    noise = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "gaussian_noise")]["confidence"]
    blur3 = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "blur_sigma3")]["confidence"]

    r_noise, _ = rank_biserial(clean, noise)
    r_blur, _ = rank_biserial(clean, blur3)

    print(f"{model}   clean vs noise             {r_noise:<15} {interpret_effect(r_noise)}")
    print(f"{model}   clean vs blur_sigma3       {r_blur:<15} {interpret_effect(r_blur)}")
    print()

print("="*60)

           EFFECT SIZES - CLEAN VIDEO CONFIDENCE            
Comparison                Effect Size     Magnitude    p-value
FP32 vs FP16              0.7728          LARGE        0.0
FP32 vs INT8              0.9747          LARGE        0.0
FP16 vs INT8              0.3918          MEDIUM       0.0

        EFFECT SIZES - CLEAN vs DEGRADED (per model)        
Comparison                     Effect Size     Magnitude
------------------------------------------------------------
FP32   clean vs noise             0.6933          LARGE
FP32   clean vs blur_sigma3       0.731           LARGE

FP16   clean vs noise             0.163           SMALL
FP16   clean vs blur_sigma3       0.1061          SMALL

INT8   clean vs noise             0.5525          LARGE
INT8   clean vs blur_sigma3       0.4636          MEDIUM



In [32]:
from scipy.stats import kruskal

conditions = ["clean", "blur_sigma1", "blur_sigma2", "blur_sigma3", "jpeg_q90", "jpeg_q70", "jpeg_q50", "gaussian_noise"]

print("="*60)
print(f"{'KRUSKAL-WALLIS TEST- ACROSS ALL CONDITIONS':^60}")
print("="*60)
print()
print("H0: confidence distributions are the same across all the conditions")
print("H1: at least one condition produces different confidence")
print()
print(f"{'Model':<8} {'H-statistic':<15} {'p-value':<20} {'Result'}")
print("-"*60)

for model in ["FP32", "FP16", "INT8"]:
    groups = []
    for condition in conditions:
        subset = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == condition)]["confidence"]
        if len(subset) > 0:
            groups.append(subset)

    stat, p = kruskal(*groups)
    result = "SIGNIFICANT" if p < 0.05 else "NOT SIGNIFICANT"
    print(f"{model:<8} {round(stat,2):<15} {p:<20} {result}")

print("="*60)
print()
print("INTERPRETATION:")
print("-"*60)
print("If SIGNIFICANT: degradation type significantly affects confidence")
print("If NOT SIGNIFICANT: all conditions produce similar confidence")


         KRUSKAL-WALLIS TEST- ACROSS ALL CONDITIONS         

H0: confidence distributions are the same across all the conditions
H1: at least one condition produces different confidence

Model    H-statistic     p-value              Result
------------------------------------------------------------
FP32     787.99          7.229295109197259e-166 SIGNIFICANT
FP16     200.87          7.503401005143058e-40 SIGNIFICANT
INT8     1173.17         4.464664980948837e-249 SIGNIFICANT

INTERPRETATION:
------------------------------------------------------------
If SIGNIFICANT: degradation type significantly affects confidence
If NOT SIGNIFICANT: all conditions produce similar confidence


In [33]:
# KRUSKAL-WALLIS FINDINGS
#
# All three models: p ≈ 0, SIGNIFICANT
# Degradation type significantly affects confidence for all models.
#
# H-statistics: INT8=1173.17, FP32=787.99, FP16=200.87
# INT8 varies most across conditions — largest clean-to-degraded gap
# FP16 varies least — most stable model across all conditions
# This confirms the effect size findings from Part 2.
#
# Conclusion: degradation type is a significant factor in model confidence
# behaviour, but its impact differs by precision level. FP16 is most
# robust to condition changes, INT8 and FP32 are more sensitive.

In [36]:
print("="*65)
print(f"{'COMPLETE STATISTICAL SUMMARY':^65}")
print("="*65)

print()
print("1. CONFIDENCE INTERVALS (95%) - CLEAN VIDEO")
print("-"*65)
print(f"{'Model':<8} {'Mean Conf':<12} {'Lower':<10} {'Upper':<10} {'Interval Width'}")
print("-"*65)

for model in ["FP32", "FP16", "INT8"]:
    subset = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "clean")]["confidence"]
    mean, lower, upper = confidence_interval(subset)
    width =  round(upper - lower, 4)
    print(f"{model:<8} {round(mean,4):<12} {lower:<10} {upper:<10} {width}")

print()
print("2. EFFECT SIZES - CLEAN VIDEO MODEL COMPARISONS")
print("-"*65)
print(f"{'Comparison':<25} {'Effect Size':<15} {'Magnitude':<12} {'p-value'}")
print("-"*65)

for name, g1, g2 in pairs:
    r, p = rank_biserial(g1, g2)
    magnitude = interpret_effect(r)
    print(f"{name:<25} {r:<15} {magnitude:<12} {p}")

print()
print("3. EFFECT SIZES - DEGRADATION IMPACT PER MODEL")
print("-"*65)
print(f"{'Comparison':<35} {'Effect Size':<15} {'Magnitude'}")
print("-"*65)

for model in ["FP32","FP16","INT8"]:
    clean = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "clean")]["confidence"]
    noise = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "gaussian_noise")]["confidence"]
    blur3 = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == "blur_sigma3")]["confidence"]

    r_noise, _ = rank_biserial(clean, noise)
    r_blur, _ = rank_biserial(clean, blur3)
    label1 = f"{model} clean vs noise"
    label2 = f"{model} clean vs blur_sigma3"
    print(f"{label1:<35} {r_noise:<15} {interpret_effect(r_noise)}")
    print(f"{label2:<35} {r_blur:<15} {interpret_effect(r_blur)}")
print()

print("4. KRUSKAL-WALLIS - ACROSS ALL CONDITIONS")
print("-"*65)
print(f"{'Model':<8} {'H-statistics':<15} {'p-value':<25} {'Result'}")
print("-"*65)

for model in ["FP32", "FP16", "INT8"]:
    group = []
    for condition in conditions:
        subset = combined_df[(combined_df["model"] == model) & (combined_df["condition"] == condition)]["confidence"]
        if len(subset) > 0:
            groups.append(subset)

    stat, p = kruskal(*groups)
    result = "SIGNIFICANT" if p < 0.05 else "NOT SIGNIFICANT"
    print(f"{model:<8} {round(stat,2):<15} {p:<25} {result}")

print()
print("5. ECE CALIBRATION SUMMARY")
print("-"*65)
print(f"{'Model':<8} {'Detection':<13} {'Correct':<10} {'Accuracy':<12} {'ECE'}")
print("-"*65)
print(f"{'FP32':<8} {642:<13} {494:<10} {'76.9%':<12} {0.181}")
print(f"{'FP16':<8} {643:<13} {497:<10} {'77.3%':<12} {0.1856}")
print(f"{'INT8':<8} {663:<13} {504:<10} {'76.0%':<12} {0.1731}")

print()
print("="*65)
print("All statistical analysis complete")
print("="*65)

                  COMPLETE STATISTICAL SUMMARY                   

1. CONFIDENCE INTERVALS (95%) - CLEAN VIDEO
-----------------------------------------------------------------
Model    Mean Conf    Lower      Upper      Interval Width
-----------------------------------------------------------------
FP32     0.3183       0.3095     0.3271     0.0176
FP16     0.5712       0.5649     0.5775     0.0126
INT8     0.6999       0.6949     0.7048     0.0099

2. EFFECT SIZES - CLEAN VIDEO MODEL COMPARISONS
-----------------------------------------------------------------
Comparison                Effect Size     Magnitude    p-value
-----------------------------------------------------------------
FP32 vs FP16              0.7728          LARGE        0.0
FP32 vs INT8              0.9747          LARGE        0.0
FP16 vs INT8              0.3918          MEDIUM       0.0

3. EFFECT SIZES - DEGRADATION IMPACT PER MODEL
-----------------------------------------------------------------
Comparison